In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
print(len(df), len(df.columns))
df.head()

5000 9


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
df['industry'].unique()

<StringArray>
[   'technology',        'retail', 'manufacturing',     'education',
    'healthcare',             nan,       'finance']
Length: 7, dtype: str

In [5]:
df['industry'].mode()[0]

'technology'

In [6]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
Name: count, dtype: int64

In [7]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

In [8]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

### Data preparation

In [9]:
str_cols = df.select_dtypes(include=['str']).columns
numerical_cols =df.select_dtypes(include=['number']).columns

In [10]:
df[str_cols] = df[str_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

In [11]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Question 1

In [12]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### Question 2

In [13]:
correlation_cols = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']
corr_matrix = df[correlation_cols].corr()
corr_matrix

,interaction_count,lead_score,number_of_courses_viewed,annual_income
interaction_count,1.000000,0.915746,0.721609,0.122842
lead_score,0.915746,1.000000,0.757204,0.229496
number_of_courses_viewed,0.721609,0.757204,1.000000,0.161300
annual_income,0.122842,0.229496,0.161300,1.000000


In [14]:
print(f"1. interaction_count & lead_score: {corr_matrix.loc['interaction_count', 'lead_score']:.4f}")
print(f"2. number_of_courses_viewed & lead_score: {corr_matrix.loc['number_of_courses_viewed', 'lead_score']:.4f}")
print(f"3. number_of_courses_viewed & interaction_count: {corr_matrix.loc['number_of_courses_viewed', 'interaction_count']:.4f}")
print(f"4. annual_income & interaction_count: {corr_matrix.loc['annual_income', 'interaction_count']:.4f}")

1. interaction_count & lead_score: 0.9157
2. number_of_courses_viewed & lead_score: 0.7572
3. number_of_courses_viewed & interaction_count: 0.7216
4. annual_income & interaction_count: 0.1228


#### Split the data

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [17]:
len(df_full_train), len(df_train), len(df_val), len(df_test)

(4000, 3000, 1000, 1000)

In [18]:
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

In [19]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [20]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [21]:
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')

In [22]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [23]:
global_converted_rate = df_full_train.converted.mean()
round(global_converted_rate, 2)

np.float64(0.57)

In [24]:
df_train.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
dtype: object

In [25]:
categorical_features = df_train.dtypes[df_train.dtypes == 'str'].index.tolist()
categorical_features

['lead_source', 'industry', 'employment_status', 'location']

In [26]:
numerical_features = df_train.select_dtypes(include=['number']).columns.tolist()
numerical_features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [27]:
df_full_train[categorical_features].nunique()

lead_source          6
industry             7
employment_status    5
location             6
dtype: int64

In [28]:
df_full_train.lead_source.unique()

<StringArray>
['events', 'organic_search', 'paid_ads', 'social_media', 'referral', 'NA']
Length: 6, dtype: str

In [29]:
df_full_train[df_full_train.lead_source == 'events'].converted.mean()

np.float64(0.4112903225806452)

In [48]:
from IPython.display import display

In [54]:
print(global_converted_rate)
for c in categorical_features:
    df_goup = df_full_train.groupby(c).converted.agg(['mean', 'count'])
    df_goup['Difference'] = round(df_goup['mean'] - global_converted_rate, 2)
    df_goup['Risk'] = round(df_goup['mean']/global_converted_rate, 2)
    display(df_goup)
    print()

0.57075


,mean,count,Difference,Risk
lead_source,,,,
NA,0.610169,118,0.04,1.07
events,0.411290,496,-0.16,0.72
organic_search,0.677579,1008,0.11,1.19
paid_ads,0.427957,930,-0.14,0.75
referral,0.763844,614,0.19,1.34
social_media,0.547962,834,-0.02,0.96


,mean,count,Difference,Risk
industry,,,,
NA,0.587629,194,0.02,1.03
education,0.500000,518,-0.07,0.88
finance,0.613402,582,0.04,1.07
healthcare,0.559398,665,-0.01,0.98
manufacturing,0.548969,388,-0.02,0.96
retail,0.553741,735,-0.02,0.97
technology,0.611111,918,0.04,1.07


,mean,count,Difference,Risk
employment_status,,,,
NA,0.580247,162,0.01,1.02
employed,0.656459,2090,0.09,1.15
self_employed,0.548276,580,-0.02,0.96
student,0.419773,617,-0.15,0.74
unemployed,0.435572,551,-0.14,0.76


,mean,count,Difference,Risk
location,,,,
NA,0.603550,169,0.03,1.06
africa,0.546154,390,-0.02,0.96
asia,0.585308,844,0.01,1.03
europe,0.584821,896,0.01,1.02
north_america,0.576923,1092,0.01,1.01
south_america,0.525452,609,-0.05,0.92


In [43]:
df_full_train.groupby('lead_source')

### Question 3

In [55]:
from sklearn.metrics import mutual_info_score

In [56]:
mutual_info_score(df_full_train.lead_source, df_full_train.converted)

0.0347609795100606

In [60]:
for c in categorical_features:
    print(f"{c}: {round(mutual_info_score(df_full_train[c], df_full_train.converted), 3)}")

lead_source: 0.035
industry: 0.003
employment_status: 0.02
location: 0.001


In [61]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [65]:
mi = df_full_train[categorical_features].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

### Question 4

In [66]:
df_full_train[numerical_features].corrwith(df_full_train.converted)

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
dtype: float64

ONE-HOT encodding

In [67]:
from sklearn.feature_extraction import DictVectorizer

In [86]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categorical_features + numerical_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical_features + numerical_features].to_dict(orient='records')
X_val = dv.transform(val_dicts)



In [94]:
from sklearn.linear_model import LogisticRegression

In [112]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [113]:
model.coef_

array([[-4.57849985e-06, -5.83434850e-04,  7.40346890e-03,
        -3.28471892e-03, -8.04069170e-03, -6.33360193e-03,
        -4.41226767e-04, -2.95759734e-03, -6.49404976e-04,
        -2.15453817e-03, -2.30751043e-03, -1.65985277e-03,
        -6.68848037e-04,  1.36728058e-01,  1.10036031e-02,
         9.51725098e-05, -8.62466509e-03,  6.94367905e-03,
        -1.37825294e-02,  8.71726672e-03, -4.18790228e-03,
         2.30545170e-04, -1.79050012e-03, -1.77472633e-03,
        -5.83764763e-04, -3.10519687e-03, -3.81533558e-03,
         5.37983416e-02]])

In [114]:
y_pred = model.predict_proba(X_val)[:, 1]


In [115]:
converted_decision = (y_pred>=0.5)

In [116]:
df_val[converted_decision]

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
682,NA,technology,employed,asia,80457.0,3,5,0.64
647,paid_ads,technology,student,africa,43944.0,2,4,0.49
878,organic_search,finance,employed,europe,73795.0,2,6,0.68
777,paid_ads,finance,NA,north_america,76619.0,1,3,0.40
2673,referral,retail,self_employed,south_america,52392.0,5,9,0.93
...,...,...,...,...,...,...,...,...
4779,organic_search,finance,employed,south_america,77662.0,4,6,0.73
2120,organic_search,retail,employed,europe,46912.0,3,6,0.66
4322,paid_ads,technology,student,north_america,0.0,1,5,0.48
2080,events,healthcare,student,north_america,40878.0,1,3,0.39


In [117]:
(y_val == converted_decision).mean()

np.float64(0.645)

In [119]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val
df_pred

,probability,prediction,actual
0,0.617257,1,1
1,0.443641,0,1
2,0.604340,1,0
3,0.645622,1,1
4,0.522142,1,0
...,...,...,...
995,0.684710,1,1
996,0.669580,1,1
997,0.458473,0,0
998,0.561770,1,0


In [124]:
df_pred['correct'] = df_pred.prediction == df_pred.actual
df_pred

,probability,prediction,actual,correct
0,0.617257,1,1,True
1,0.443641,0,1,False
2,0.604340,1,0,False
3,0.645622,1,1,True
4,0.522142,1,0,False
...,...,...,...,...
995,0.684710,1,1,True
996,0.669580,1,1,True
997,0.458473,0,0,True
998,0.561770,1,0,False


In [125]:
df_pred.correct.mean()

np.float64(0.645)

### Question 5

In [128]:
all_features = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 
                'annual_income', 'lead_source', 'employment_status', 'industry', 'location']

baseline_accuracy = (y_val == converted_decision).mean()

for feature in all_features:
    subset_features = [f for f in all_features if f != feature]
    
    train_dicts = df_train[subset_features].to_dict(orient='records')
    X_train_subset = dv.transform(train_dicts)
    
    val_dicts = df_val[subset_features].to_dict(orient='records')
    X_val_subset = dv.transform(val_dicts)
    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_subset, y_train)
    
    y_pred = model.predict(X_val_subset)
    subset_accuracy = (y_pred == y_val).mean()
    
    difference = baseline_accuracy - subset_accuracy
    
    print(f"Excluding '{feature}': New Accuracy = {subset_accuracy:.4f} | Diff = {difference:.4f}")


Excluding 'interaction_count': New Accuracy = 0.6010 | Diff = 0.0440
Excluding 'lead_score': New Accuracy = 0.6440 | Diff = 0.0010
Excluding 'number_of_courses_viewed': New Accuracy = 0.6430 | Diff = 0.0020
Excluding 'annual_income': New Accuracy = 0.7240 | Diff = -0.0790
Excluding 'lead_source': New Accuracy = 0.6420 | Diff = 0.0030
Excluding 'employment_status': New Accuracy = 0.6450 | Diff = 0.0000
Excluding 'industry': New Accuracy = 0.6450 | Diff = 0.0000
Excluding 'location': New Accuracy = 0.6450 | Diff = 0.0000


### Question 6